# 2. Análise: seleção de variáveis e agrupamento (RMSP)

Notebook do artigo sobre tipologias de precariedade habitacional, aplicado aos 39 municípios da Região Metropolitana de São Paulo, com dados do Censo Demográfico 2022 (IBGE), Agregados por Setores Censitários consolidados por município.

**Estrutura:**
- **Preparação dos dados (etapas 1 a 10):** estatística descritiva, correlações, seleção das variáveis, conversão em taxas, variáveis suplementares, redundância e normalização.
- **Agrupamento (etapa 11):** escolha do número de grupos, estabilidade, modelo final, silhueta e teste de sensibilidade.
- **Perfil e localização dos grupos (etapa 12):** médias dos grupos, mapa e densidade.
- **Variáveis suplementares de baixa ocorrência (etapa 13).**
- **População em favelas e comunidades urbanas (etapa 14).**

As células descrevem apenas o que cada etapa faz. A interpretação dos resultados está no artigo.

In [ ]:
# Library versions used in this run (copy them into requirements.txt to pin the environment)
import importlib
for lib in ["pandas", "numpy", "sklearn", "scipy", "matplotlib", "seaborn", "plotly", "geopandas"]:
    try:
        print(f"{lib}=={importlib.import_module(lib).__version__}")
    except ImportError:
        print(f"{lib}: not installed")

In [ ]:
# 1. LOAD CONSOLIDATED DATA FROM GITHUB

import pandas as pd

# Processed files generated by 1_BD_consolidacao.ipynb (1_DATA/processed)
path = "https://raw.githubusercontent.com/julianevesa/rmsp-habitacao-censo2022/main/1_DATA/processed/"

# Load the data directly from GitHub
df = pd.read_csv(path + "base_final_SP.csv", sep=";")

## Recorte espacial

Mantém apenas os 39 municípios da RMSP. Todas as etapas seguintes são calculadas só para esses municípios.

In [ ]:
import unicodedata

SCOPE = "RMSP"  # This repository refers only to the São Paulo Metropolitan Region (39 municipalities)

RMSP = [
    "Arujá", "Barueri", "Biritiba Mirim", "Caieiras", "Cajamar", "Carapicuíba",
    "Cotia", "Diadema", "Embu das Artes", "Embu-Guaçu", "Ferraz de Vasconcelos",
    "Francisco Morato", "Franco da Rocha", "Guararema", "Guarulhos",
    "Itapecerica da Serra", "Itapevi", "Itaquaquecetuba", "Jandira", "Juquitiba",
    "Mairiporã", "Mauá", "Mogi das Cruzes", "Osasco", "Pirapora do Bom Jesus",
    "Poá", "Ribeirão Pires", "Rio Grande da Serra", "Salesópolis", "Santa Isabel",
    "Santana de Parnaíba", "Santo André", "São Bernardo do Campo",
    "São Caetano do Sul", "São Lourenço da Serra", "São Paulo", "Suzano",
    "Taboão da Serra", "Vargem Grande Paulista",
]

def norm(name):
    s = unicodedata.normalize("NFD", str(name)).encode("ascii", "ignore").decode()
    return s.lower().replace("-", " ").strip()

rmsp_norm = {norm(n) for n in RMSP}
df = df[df["NM_MUN"].map(norm).isin(rmsp_norm)].reset_index(drop=True)
found = {norm(n) for n in df["NM_MUN"]}
missing = [n for n in RMSP if norm(n) not in found]
print(f"RMSP municipalities found: {len(df)} of 39")
if missing:
    print("Not found (check spelling in NM_MUN):", missing)
assert len(df) == 39, "The RMSP must have 39 municipalities."

## 1. Estatística descritiva de todas as variáveis

Média, mínimo, máximo, valores nulos e zeros de todas as variáveis numéricas da base.

In [ ]:
import numpy as np
import pandas as pd

# 1. Filters the dataframe, keeping only the numeric columns
df_num = df.select_dtypes(include=[np.number])

# 2. Creates the basic statistics table
estatistica = df_num.describe().T

estatistica["qtd_nulos"] = df_num.isnull().sum()
estatistica["qtd_zeros"] = (df_num == 0).sum()

display(
    estatistica[
        ["count", "mean", "min", "max", "qtd_nulos", "qtd_zeros"]
    ].round(2)
)

## 2. Correlação inicial entre todas as variáveis

Correlação de Pearson entre todas as variáveis, em valores absolutos. Os pares com |r| ≥ 0,85 são salvos em `initial_variable_correlation_analysis.csv`. Essa etapa mapeia redundâncias, como a mesma condição repetida em diferentes tabelas do IBGE.

In [ ]:
import numpy as np
import pandas as pd

# 1. Calculates the correlation matrix
correlation_matrix = df_num.corr(method="pearson")

# 2. Converts the matrix into a list of pairs
df_pairs = (
    correlation_matrix.where(
        np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
    )
    .stack()
    .reset_index()
)
df_pairs.columns = ["Variable A", "Variable B", "Correlation"]

# 3. Filters strong correlations
threshold = 0.85
df_screening = df_pairs[df_pairs["Correlation"].abs() >= threshold].copy()

# 4. Defines the Relationship Type
df_screening["Relationship Type"] = np.where(
    df_screening["Correlation"] >= threshold,
    "Direct (Same Direction)",
    "Inverse (Mirror)",
)

# 5. Saves the final dataset locally
df_screening = df_screening.sort_values(by="Correlation", ascending=False).round(
    2
)
df_screening.to_csv(
    "initial_variable_correlation_analysis.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False,
)

### Contagem das correlações iniciais

Número de variáveis com |r| ≥ 0,85 com pelo menos uma outra e com o total de domicílios (v0007), usado no texto do artigo.

In [ ]:
pairs = pd.read_csv("initial_variable_correlation_analysis.csv", sep=";", encoding="utf-8-sig")
strong = pairs[pairs["Correlation"].abs() >= 0.85]
vars_strong = set(strong["Variable A"]) | set(strong["Variable B"])
with_total = strong[(strong["Variable A"] == "v0007") | (strong["Variable B"] == "v0007")]
print(f"Municípios na base: {len(df_num)}")
print(f"Variáveis com |r| >= 0,85 com pelo menos uma outra: {len(vars_strong - {'v0007'})}")
print(f"Variáveis com |r| >= 0,85 com o total de domicílios (v0007): {len(with_total)}")
print(f"Total de colunas numéricas analisadas (inclui CD_MUN): {df_num.shape[1]}")

## 3. Pré-seleção das variáveis

Variáveis selecionadas pelos critérios conceituais: dimensões da moradia adequada (água, esgoto e banheiro, durabilidade da moradia e resíduos sólidos), apenas variáveis de privação e apenas variáveis para o total de domicílios (sem desagregação por atributos do responsável). Os critérios e as exclusões estão no dicionário de variáveis, mais abaixo, e em `docs/selecao_variaveis.md`.

In [ ]:
import pandas as pd

# 1. List of deprivation variables (IBGE)
pre_selected_variables = [
    "V00050",
    "V00051",
    "V00052",
    "V00053",
    "V00054",
    "V00055",
    "V00056",
    "V00057",
    "V00058",
    "V00113",
    "V00114",
    "V00115",
    "V00116",
    "V00117",
    "V00118",
    "V00200",
    "V00201",
    "V00236",
    "V00237",
    "V00312",
    "V00313",
    "V00314",
    "V00315",
    "V00316",
    "V00399",
    "V00400",
    "V00401",
    "V00402",
    "V00464",
]

# 2. Generates, filters, renames, and rounds the descriptive statistics
df_descriptive = (
    df[pre_selected_variables]
    .describe()
    .T[["count", "mean", "std", "min", "max"]]
    .rename(
        columns={
            "count": "Count",
            "mean": "Mean",
            "std": "Standard Deviation",
            "min": "Minimum",
            "max": "Maximum",
        }
    )
    .round(2)
)

# 3. Saves the header and the table directly to a local TXT file
txt_output = "descriptive_statistics_pre_selection_table.txt"
with open(txt_output, "w", encoding="utf-8") as f:
    f.write(
        "DESCRIPTIVE STATISTICS TABLE (RAW VARIABLES) - PRE-SELECTED.\n\n"
    )
    f.write(df_descriptive.to_string())

# 4. Displays results
display(df_descriptive)

## 4. Correlação das variáveis pré-selecionadas (valores absolutos)

Correlação entre as variáveis pré-selecionadas, ainda em contagens. Serve para verificar o efeito de escala: em valores absolutos, a correlação tende a refletir o tamanho dos municípios.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 1. Filters the variable list removing V00051
filtered_variables = [v for v in pre_selected_variables if v != "V00051"]

# 2. Calculates the correlation matrix
correlation_matrix = df[filtered_variables].corr(method="pearson")

# 3. Extracts, sorts, and formats the correlation pairs
df_pairs = (
    correlation_matrix.where(
        np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
    )
    .stack()
    .reset_index()
)
df_pairs.columns = ["Variable A", "Variable B", "Correlation"]
df_pairs = df_pairs.sort_values(by="Correlation", ascending=False)

# 4. Saves results directly to a local TXT file
txt_output = "raw_variables_correlation_list.txt"
with open(txt_output, "w", encoding="utf-8") as f:
    f.write("=== RAW VARIABLES CORRELATION LIST ===\n")
    f.write("Note: Variable V00051 was removed due to lack of variance.\n\n")
    f.write(
        df_pairs.to_string(
            index=False, formatters={"Correlation": "{:.4f}".format}
        )
    )

print(f"TXT file saved to: {txt_output}")

# 5. Configures and generates the Heatmap
plt.figure(figsize=(32, 26))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(
    correlation_matrix,
    mask=mask,
    cmap="coolwarm",
    vmax=1.0,
    vmin=-1.0,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.7, "label": "Pearson Correlation Coefficient"},
)

plt.title(
    "Pearson Correlation Matrix - Raw Census Variables",
    fontsize=24,
    fontweight="bold",
    pad=20,
)
plt.xticks(rotation=45, ha="right", fontsize=12)
plt.yticks(rotation=0, fontsize=12)
plt.tight_layout()

# 6. Saves and displays the plot locally
plot_output = "raw_variables_correlation_heatmap.png"
plt.savefig(plot_output, dpi=300, bbox_inches="tight")
print(f"Plot successfully saved to: {plot_output}")

plt.show()

## 5. Conversão em taxas

As contagens são convertidas em porcentagens:
- variáveis de serviços e adequação da moradia: divididas pelo total de domicílios particulares permanentes ocupados (V00001), pois o IBGE só levanta essas informações para esses domicílios;
- tipos de domicílio improvisado: divididos pelo total de domicílios ocupados (v0007).

A V00051 é retirada por não ter ocorrência no recorte.

In [ ]:
import pandas as pd

# 1. Household variables converted to rates = pre-selected variables,
#    excluding those with no occurrence in the study area (all zeros)
no_variance = ["V00051"]
household_variables = [v for v in pre_selected_variables if v not in no_variance]

# 2a. Permanent dwellings (DPPO): services and housing adequacy / total DPPO (V00001)
dpio_types = ["V00053", "V00054", "V00055", "V00056", "V00057", "V00058"]
dppo_vars = [v for v in household_variables if v not in dpio_types]
df_rates = df[dppo_vars].div(df["V00001"], axis=0) * 100

# 2b. Improvised dwelling types (DPIO) / total occupied dwellings (v0007)
df_rates[dpio_types] = df[dpio_types].div(df["v0007"], axis=0) * 100
df_rates.columns = [f"{v}_P" for v in df_rates.columns]


# 4. Inserts identifiers at the beginning of the DataFrame
df_rates.insert(0, "CD_MUN", df["CD_MUN"].values)
df_rates.insert(1, "NM_MUN", df["NM_MUN"].values)

if "NM_UF" in df.columns:
    df_rates.insert(2, "NM_UF", df["NM_UF"].values)
if "CD_UF" in df.columns:
    df_rates.insert(3, "CD_UF", df["CD_UF"].values)

# 5. Generates the descriptive statistics table
rates_descriptive = (
    df_rates.describe()
    .T[["count", "mean", "std", "min", "max"]]
    .rename(
        columns={
            "count": "Count",
            "mean": "Mean (%)",
            "std": "Standard Deviation",
            "min": "Minimum (%)",
            "max": "Maximum (%)",
        }
    )
    .round(2)
)

# 6. Saves the descriptive statistics table directly to a local TXT file
txt_output = "selected_variables_rates_statistics_table.txt"
with open(txt_output, "w", encoding="utf-8") as f:
    f.write(
        "DESCRIPTIVE STATISTICS TABLE (PROPORTIONS (%))\n"
    )
    f.write(
        "DPPO variables deflated by V00001; improvised dwelling types by v0007.\n\n"
    )
    f.write(rates_descriptive.to_string())

# 7. Displays results
display(rates_descriptive)

### Verificação da sobreposição da V00463

As formas principais de abastecimento fora da rede (V00112 a V00118) devem corresponder aos domicílios sem ligação (V00464) mais os ligados que usam principalmente outra forma (V00463).

In [ ]:
check = df[["V00112","V00113","V00114","V00115","V00116","V00117","V00118"]].sum(axis=1)
diff = check - (df["V00463"] + df["V00464"])
rel = (diff.abs() / (df["V00463"] + df["V00464"]) * 100).round(2)
print(f"Diferença relativa: mediana {rel.median()}% | máxima {rel.max()}%")

# 📑 Dicionário das variáveis selecionadas — Meta 11.1 dos ODS (Censo 2022)

Variáveis do Censo Demográfico 2022 (IBGE), Agregados por Setores Censitários (características dos domicílios), consolidadas por município. Foram mantidas apenas variáveis que indicam **déficit, privação ou inadequação** do domicílio: valores mais altos sempre significam maior privação.

---

### 1. Identificadores e denominadores (não usados no agrupamento)
* **`CD_MUN`**: Código oficial do município (7 dígitos - IBGE).
* **`V00001`**: Domicílios particulares permanentes ocupados (DPPO). Denominador das variáveis de serviços e de adequação da moradia, que o IBGE levanta apenas para os domicílios permanentes.
* **`v0007`** (V0007, arquivo Básico): Total de domicílios particulares ocupados (permanentes e improvisados). Denominador dos tipos de domicílio improvisado.

---

### 2. Adequação da moradia e domicílios improvisados
* **`V00050`**: Casa de cômodos ou cortiço.
* **`V00051`**: Habitação indígena sem paredes ou maloca. **Excluída:** sem ocorrência no recorte.
* **`V00052`**: Estrutura residencial permanente degradada ou inacabada.
* **`V00053`**: Domicílio improvisado: tenda ou barraca de lona, plástico ou tecido.
* **`V00054`**: Domicílio improvisado: dentro de estabelecimento em funcionamento. **Suplementar:** prevalência máxima abaixo de 0,07% (8 municípios; máximo de 0,011%).
* **`V00055`**: Domicílio improvisado: abrigo natural e outras estruturas improvisadas. **Suplementar:** prevalência máxima abaixo de 0,07% (15 municípios; máximo de 0,048%).
* **`V00056`**: Domicílio improvisado: estrutura improvisada em logradouro público (exceto tenda ou barraca).
* **`V00057`**: Domicílio improvisado: estrutura não residencial permanente degradada ou inacabada. **Suplementar:** prevalência máxima abaixo de 0,07% (6 municípios; máximo de 0,043%).
* **`V00058`**: Domicílio improvisado: veículo (carro, caminhão, trailer, barco etc.). **Suplementar:** ocorre em apenas 1 município.

---

### 3. Abastecimento de água
* **`V00113`**: Utiliza poço raso, freático ou cacimba.
* **`V00114`**: Utiliza fonte, nascente ou mina.
* **`V00115`**: Utiliza carro-pipa.
* **`V00116`**: Utiliza água da chuva armazenada.
* **`V00117`**: Utiliza rios, açudes, córregos, lagos e igarapés.
* **`V00118`**: Utiliza outra forma de abastecimento de água.
* **`V00200`**: Água chega encanada, mas apenas ao terreno.
* **`V00201`**: Água não chega encanada ao domicílio.
* **`V00464`**: Domicílio não possui ligação à rede geral de distribuição de água. **Retirada por redundância:** sobrepõe-se às formas de abastecimento fora da rede geral (V00112 a V00118), com r = 0,96 com a V00113.

---

### 4. Esgotamento sanitário e banheiro
* **`V00236`**: Apenas banheiro de uso comum a mais de um domicílio.
* **`V00237`**: Apenas sanitário ou buraco para dejeções. **Suplementar:** prevalência máxima abaixo de 0,07% (11 municípios; máximo de 0,063%).
* **`V00312`**: Esgotamento por fossa rudimentar ou buraco.
* **`V00313`**: Esgotamento por vala.
* **`V00314`**: Esgotamento em rio, lago, córrego ou mar.
* **`V00315`**: Outra forma de esgotamento.
* **`V00316`**: Sem esgotamento, por não ter banheiro nem sanitário. **Suplementar:** prevalência máxima abaixo de 0,07% (10 municípios; máximo de 0,016%).

---

### 5. Resíduos sólidos
* **`V00399`**: Lixo queimado na propriedade.
* **`V00400`**: Lixo enterrado na propriedade.
* **`V00401`**: Lixo jogado em terreno baldio, encosta ou área pública.
* **`V00402`**: Outro destino do lixo.

---

> 💡 **Notas metodológicas sobre exclusões e variáveis suplementares:**
> * **Variáveis de adequação e cobertura** (V00111, V00112, V00199, V00309, V00310, V00311, V00397 e V00398) foram excluídas por expressarem a presença do serviço ou uma forma adequada de atendimento, e não a privação.
> * **Variáveis de conforto** (V00232 a V00235 e V00485 a V00494, número de banheiros) foram excluídas por não representarem privação.
> * **V00238** foi excluída por registrar a mesma condição da V00316 em outra tabela do IBGE (r ≈ 1).
> * **V00463** (domicílio ligado à rede geral, mas que utiliza principalmente outra forma) foi excluída porque esses domicílios já estão contados nas formas principais de abastecimento (V00112 a V00118). As somas por município apresentaram apenas pequenas diferenças (mediana de 1,8%; máximo de 6,4%), atribuíveis a dados omitidos por sigilo.
> * **V00006 e V00009** (moradores e crianças em domicílios improvisados) foram excluídas por medirem o mesmo fenômeno dos tipos de domicílio improvisado, em outra unidade (pessoas).
> * **Variáveis desagregadas por atributos da pessoa responsável** (sexo, cor ou raça) foram excluídas; manteve-se apenas a variável referente ao total de domicílios.
> * **Ocorrência mínima:** variáveis presentes em menos de 5 municípios ou com prevalência máxima abaixo de 0,07% dos domicílios não foram usadas no agrupamento e foram analisadas como **variáveis suplementares**.
> * **População em favelas e comunidades urbanas (N_POP_DPPO)** provém de base específica do IBGE e não foi usada no agrupamento; é analisada apenas como variável suplementar.

## 6. Correlação entre as taxas

Correlação de Pearson entre as variáveis já convertidas em porcentagens.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 1. Selects columns ending with '_P' and removes 'V00058_P'
final_rates = [
    t for t in df_rates.columns if t.endswith("_P") and t != "V00058_P"
]

# 2. Calculates the new correlation matrix for rates
rates_correlation_matrix = df_rates[final_rates].corr(method="pearson")

# 3. Extracts and sorts correlation pairs
df_pairs = (
    rates_correlation_matrix.where(
        np.triu(np.ones(rates_correlation_matrix.shape), k=1).astype(bool)
    )
    .stack()
    .reset_index()
)
df_pairs.columns = ["Variable A", "Variable B", "Correlation"]
df_pairs = df_pairs.sort_values(by="Correlation", ascending=False)

# 4. Saves the correlation list directly to a local TXT file
txt_output = "final_rates_correlation_list.txt"
with open(txt_output, "w", encoding="utf-8") as f:
    f.write("CORRELATION LIST FOR PERCENTAGE RATES (v0007)\n")
    f.write(
        "Analysis of real relationships after removing the demographic scale effect.\n\n"
    )
    f.write(
        df_pairs.to_string(
            index=False, formatters={"Correlation": "{:.4f}".format}
        )
    )

print(f"TXT file successfully saved to: {txt_output}")

# 5. Configures and generates the Heatmap
plt.figure(figsize=(30, 24))
mask = np.triu(np.ones_like(rates_correlation_matrix, dtype=bool))

sns.heatmap(
    rates_correlation_matrix,
    mask=mask,
    cmap="coolwarm",
    vmax=1.0,
    vmin=-1.0,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.7, "label": "Pearson Correlation Coefficient"},
)

plt.title(
    "Correlation Matrix - Unified Percentage Rates (%)",
    fontsize=22,
    fontweight="bold",
    pad=20,
)
plt.xticks(rotation=45, ha="right", fontsize=11)
plt.yticks(rotation=0, fontsize=11)
plt.tight_layout()

# 6. Saves and displays the plot locally
plot_output = "final_rates_correlation_heatmap.png"
plt.savefig(plot_output, dpi=300, bbox_inches="tight")
print(f"Plot successfully saved to: {plot_output}")

plt.show()

## 7. Ocorrência das variáveis

Número de municípios em que cada condição aparece e valor máximo (%).

In [ ]:
# Number of municipalities where each condition occurs (rate > 0)
rate_cols = [c for c in df_rates.columns if c.endswith("_P")]
occurrence = pd.DataFrame({
    "Municipalities with occurrence": (df_rates[rate_cols] > 0).sum(),
    "Max (%)": df_rates[rate_cols].max().round(3),
}).sort_values("Municipalities with occurrence")
display(occurrence)

## 8. Variáveis suplementares

Variáveis presentes em menos de 5 municípios ou com prevalência máxima abaixo de 0,07% dos domicílios não entram no agrupamento. Elas são analisadas depois, como suplementares (etapa 13).

In [ ]:
N_MIN = 5          # minimum number of municipalities
PREV_MIN = 0.07    # minimum maximum prevalence (% of households)
supplementary_vars = occurrence[
    (occurrence["Municipalities with occurrence"] < N_MIN)
    | (occurrence["Max (%)"] < PREV_MIN)
].index.tolist()
print(f"Supplementary (not clustered): {supplementary_vars}")

## 9. Redundância entre as taxas

Pares de taxas com |r| ≥ 0,85. Retira-se a variável quando uma contém a outra (relação de parte e todo). A V00464 (sem ligação à rede de água) é retirada por ser o total das formas de abastecimento fora da rede (V00113 a V00118).

In [ ]:
import numpy as np

# 1. Pairs of rates with |r| >= 0.85 (same threshold used in the initial screening)
threshold = 0.85
corr_rates = df_rates[final_rates].corr(method="pearson")
pairs_rates = (
    corr_rates.where(np.triu(np.ones(corr_rates.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
pairs_rates.columns = ["Variable A", "Variable B", "Correlation"]
redundant_pairs = pairs_rates[pairs_rates["Correlation"].abs() >= threshold]
display(redundant_pairs.sort_values("Correlation", ascending=False).round(4))

# 2. DEFINE here which variables to remove after inspecting the pairs above
redundant_to_remove = ["V00464_P"]
print(f"Variables removed for redundancy: {redundant_to_remove}")

## 10. Normalização Min-Max

As taxas das variáveis do agrupamento são transformadas para a escala de 0 a 1, para que todas tenham o mesmo peso no cálculo das distâncias.

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Filter columns ending with "_P" directly from the source dataframe
rate_columns = [
    c for c in df_rates.columns
    if c.endswith("_P") and c not in redundant_to_remove + supplementary_vars
]
print(f"Variables used in the clustering: {len(rate_columns)}")

# 2. Normalize data using CD_MUN as a temporary index to keep rows aligned
df_rates_normalized = pd.DataFrame(
    MinMaxScaler().fit_transform(df_rates.set_index("CD_MUN")[rate_columns]),
    columns=rate_columns,
    index=df_rates["CD_MUN"]
).reset_index()

# 3. Save the normalized dataset locally and preview the output
df_rates_normalized.to_csv("df_rates_normalized.csv", index=False, encoding="utf-8")
display(df_rates_normalized.head())

## 11. Escolha do número de grupos (k)

Para k de 2 a 10: inércia (método do cotovelo) e coeficiente de silhueta.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Load data, setting CD_MUN as index to isolate it from KMeans features
df_filtered = pd.read_csv("df_rates_normalized.csv", index_col="CD_MUN")

# 1. Run K-Means and compute metrics
k_range = range(2, 11)
models = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(df_filtered) for k in k_range]

inertias = [model.inertia_ for model in models]
silhouette_scores = [silhouette_score(df_filtered, model.labels_) for model in models]

# 2. Configure dual-axis plot (Elbow vs. Silhouette)
fig, ax1 = plt.subplots(figsize=(10, 5))

# Inertia Plot (Left Axis - Blue)
ax1.set_xlabel("Number of Clusters (K)", fontsize=11, fontweight="bold")
ax1.set_ylabel("Inertia", color="tab:blue", fontsize=11)
ax1.plot(k_range, inertias, marker="o", color="tab:blue", linewidth=2)
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.grid(True, linestyle="--", alpha=0.5)

# Silhouette Score Plot (Right Axis - Red)
ax2 = ax1.twinx()
ax2.set_ylabel("Silhouette Coefficient", color="tab:red", fontsize=11)
ax2.plot(k_range, silhouette_scores, marker="s", color="tab:red", linewidth=2, linestyle="--")
ax2.tick_params(axis="y", labelcolor="tab:red")

plt.title("Cluster Selection: Elbow Method vs. Silhouette Score", fontsize=12, fontweight="bold", pad=10)
fig.tight_layout()

# 3. Save and display the plot
plt.savefig("elbow_silhouette_selection_plot.png", dpi=300, bbox_inches="tight")
plt.show()

# 4. Display numerical results
display(pd.DataFrame({"K": k_range, "Inertia": inertias, "Silhouette": silhouette_scores}).round(4))

### Estabilidade por número de grupos

Para k de 2 a 7: concordância entre 50 rodadas com sementes diferentes (ARI), concordância com o agrupamento hierárquico de Ward e tamanho do menor grupo.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score
rows = []
for k in range(2, 8):
    base = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(df_filtered)
    seeds = [adjusted_rand_score(base, KMeans(n_clusters=k, random_state=s, n_init=10).fit_predict(df_filtered)) for s in range(50)]
    ward = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(df_filtered)
    rows.append({"K": k, "ARI seeds (mean)": np.mean(seeds), "ARI seeds (min)": np.min(seeds),
                 "ARI Ward": adjusted_rand_score(base, ward), "Smallest cluster": np.bincount(base).min()})
display(pd.DataFrame(rows).round(3))

### Modelo final

K-Means com o k escolhido, semente fixa (`random_state=42`) e 10 inicializações.

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans

# 1. Load normalized data, isolating CD_MUN from features by setting it as index
df_filtered = pd.read_csv("df_rates_normalized.csv", index_col="CD_MUN")

# 2. Instantiate and fit the final K-Means model (adjust n_clusters as needed)
optimal_k = 2  # DEFINE after reviewing the new Elbow/Silhouette results above (and justify the choice in the text)
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(df_filtered)

# 3. Append clusters to the dataset and bring CD_MUN back as a regular column
df_final = df_filtered.copy()
df_final["cluster"] = clusters
df_final = df_final.reset_index()

# 4. Save the final clustered dataset locally and preview results
df_final.to_csv("df_census_clusters.csv", index=False, encoding="utf-8")
display(df_final.head())

### Testes de estabilidade do modelo final

1. ARI entre o modelo final e 50 rodadas com sementes diferentes (1 = grupos idênticos).
2. Número de municípios por grupo.
3. Silhueta média de cada grupo.
4. Comparação com o agrupamento hierárquico de Ward (ARI e tabela cruzada) e dendrograma.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_samples

X = df_filtered.values

# 1. Stability across seeds
seed_labels = [
    KMeans(n_clusters=optimal_k, random_state=s, n_init=10).fit_predict(X)
    for s in range(50)
]
ari_seeds = np.array([adjusted_rand_score(clusters, lab) for lab in seed_labels])
print(f"1. Seeds - ARI vs. final model: mean {ari_seeds.mean():.3f} | "
      f"min {ari_seeds.min():.3f} | runs identical (ARI = 1): {(ari_seeds == 1).sum()} of 50")

# 2. Cluster sizes
print("\n2. Municipalities per cluster:")
print(pd.Series(clusters).value_counts().sort_index().to_string())

# 3. Silhouette by cluster
sil = silhouette_samples(X, clusters)
sil_by_cluster = pd.DataFrame({"cluster": clusters, "silhouette": sil}).groupby("cluster")["silhouette"].mean().round(3)
print(f"\n3. Mean silhouette by cluster (overall: {sil.mean():.3f}):")
print(sil_by_cluster.to_string())

# 4. Ward's hierarchical clustering
ward_labels = AgglomerativeClustering(n_clusters=optimal_k, linkage="ward").fit_predict(X)
print(f"\n4. Ward vs. K-Means - ARI: {adjusted_rand_score(clusters, ward_labels):.3f}")
crosstab = pd.crosstab(pd.Series(clusters, name="K-Means"), pd.Series(ward_labels, name="Ward"))
display(crosstab)

# Dendrogram (labels with municipality names)
names = df.set_index("CD_MUN").loc[df_filtered.index, "NM_MUN"].astype(str).values
plt.figure(figsize=(14, 6))
dendrogram(linkage(X, method="ward"), labels=names, leaf_rotation=90, leaf_font_size=9)
plt.title("Ward's Hierarchical Clustering - Dendrogram", fontsize=13, fontweight="bold")
plt.ylabel("Distance (Ward)")
plt.tight_layout()
plt.savefig("ward_dendrogram.png", dpi=300, bbox_inches="tight")
plt.show()

# Save a summary table
pd.DataFrame({
    "Check": ["ARI seeds (mean)", "ARI seeds (min)", "Silhouette (overall)", "ARI Ward vs K-Means"],
    "Value": [ari_seeds.mean(), ari_seeds.min(), sil.mean(), adjusted_rand_score(clusters, ward_labels)],
}).round(3).to_csv("stability_summary.csv", index=False, encoding="utf-8")

### Silhueta por município

Valores perto de 1: município característico do grupo; perto de 0: na fronteira entre os grupos; negativos: mais parecido, em média, com o outro grupo.

In [ ]:
from sklearn.metrics import silhouette_samples
sil_mun = pd.DataFrame({"CD_MUN": df_filtered.index.astype(int), "cluster": clusters,
                        "silhouette": silhouette_samples(df_filtered.values, clusters).round(3)})
sil_mun = sil_mun.merge(df[["CD_MUN", "NM_MUN"]].astype({"CD_MUN": int}), on="CD_MUN")
display(sil_mun.sort_values(["cluster", "silhouette"]))
sil_mun.to_csv("silhouette_by_municipality.csv", index=False, encoding="utf-8-sig")

### Teste de sensibilidade

Repete o agrupamento incluindo as variáveis de baixa ocorrência (27 variáveis, frente a 21 na análise principal) e compara os dois resultados pelo ARI e pela silhueta. Em seguida, lista os municípios que mudam de grupo.

In [ ]:
# Teste de sensibilidade: agrupamento com todas as 27 variáveis (sem o critério de baixa ocorrência)
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import MinMaxScaler

all_vars = [c for c in df_rates.columns if c.endswith("_P") and c not in redundant_to_remove]
X_all = MinMaxScaler().fit_transform(df_rates.set_index("CD_MUN").loc[df_filtered.index, all_vars])
labels_all = KMeans(n_clusters=optimal_k, random_state=42, n_init=10).fit_predict(X_all)

print(f"Variáveis sem o corte: {len(all_vars)} | com o corte: {df_filtered.shape[1]}")
print(f"ARI entre os dois agrupamentos: {adjusted_rand_score(clusters, labels_all):.3f}")
print(f"Silhueta sem o corte: {silhouette_score(X_all, labels_all):.3f} | com o corte: {silhouette_score(df_filtered, clusters):.3f}")

In [ ]:
comp = pd.DataFrame({"CD_MUN": df_filtered.index.astype(int), "com_corte": clusters, "sem_corte": labels_all})
display(pd.crosstab(comp["com_corte"], comp["sem_corte"]))
mapa_rotulos = pd.crosstab(comp["com_corte"], comp["sem_corte"]).idxmax(axis=0).to_dict()
comp["sem_corte_alinhado"] = comp["sem_corte"].map(mapa_rotulos)
mudaram = comp[comp["com_corte"] != comp["sem_corte_alinhado"]].merge(df[["CD_MUN", "NM_MUN"]].astype({"CD_MUN": int}), on="CD_MUN")
print(mudaram[["NM_MUN", "com_corte", "sem_corte_alinhado"]])

## 12. Perfil e localização dos grupos

### Perfil dos grupos (centroides normalizados)

Média de cada variável em cada grupo, na escala normalizada (0 a 1), ordenada pela disparidade entre os grupos (desvio-padrão das médias).

In [ ]:
import pandas as pd

# PANDAS CONFIGURATION: Force full display without truncating rows
pd.set_option("display.max_rows", None)

# 1. Load the local cluster results from your actual previous cell output
df_results = pd.read_csv("df_census_clusters.csv")

# 2. Automatically select rate columns (ending in _P) and the cluster column
rate_columns = [c for c in df_results.columns if c.endswith("_P")]
df_rates_cluster = df_results[rate_columns + ["cluster"]]

# 3. DYNAMIC CENTROID PROFILE CONSTRUCTION (Adapts automatically to any K)
df_centroids_real = df_rates_cluster.groupby("cluster").mean().T

# Automatically generate column names based on the chosen optimal_k (e.g., Cluster_0_Mean, Cluster_1_Mean...)
mean_columns = [f"Cluster_{i}_Mean" for i in range(optimal_k)]
df_centroids_real.columns = mean_columns

# 4. MULTICLUSTER SORTING CRITERION (Intergroup Disparity)
# Calculates standard deviation across all group means dynamically
df_centroids_real["Intergroup_Disparity"] = df_centroids_real[mean_columns].std(axis=1)

# Sort the report in descending order based on the magnitude of disparity
df_centroids_real = df_centroids_real.sort_values(
    by="Intergroup_Disparity", ascending=False
)

# 5. Save the final centroids analysis dataset locally with a dynamic filename
output_centroids_csv = f"centroids_analysis_{optimal_k}_clusters.csv"
df_centroids_real.to_csv(output_centroids_csv, encoding="utf-8")
print(f"✅ Centroid profiles analysis saved as '{output_centroids_csv}'")

# 6. DESCRIÇÃO DAS VARIÁVEIS (português)
legend_dict_pt = {
    "V00050_P": "DPPO Tipo - Casa de cômodos ou cortiço",
    "V00051_P": "DPPO Tipo - Habitação indígena sem paredes ou maloca",
    "V00052_P": "DPPO Tipo - Estrutura residencial permanente degradada ou inacabada",
    "V00053_P": "DPIO Tipo - Tenda ou barraca de lona, plástico ou tecido",
    "V00054_P": "DPIO Tipo - Dentro de estabelecimento em funcionamento",
    "V00055_P": "DPIO Tipo - Abrigo natural e outras estruturas improvisadas",
    "V00056_P": "DPIO Tipo - Estrutura improvisada em logradouro público",
    "V00057_P": "DPIO Tipo - Estrutura não residencial degradada ou inacabada",
    "V00058_P": "DPIO Tipo - Veículo (carro, caminhão, trailer, barco etc.)",
    "V00113_P": "DPPO Água - Poço raso, freático ou cacimba",
    "V00114_P": "DPPO Água - Fonte, nascente ou mina",
    "V00115_P": "DPPO Água - Carro-pipa",
    "V00116_P": "DPPO Água - Água da chuva armazenada",
    "V00117_P": "DPPO Água - Rios, açudes, córregos, lagos e igarapés",
    "V00118_P": "DPPO Água - Outra forma de abastecimento",
    "V00200_P": "DPPO Água - Encanada apenas até o terreno",
    "V00201_P": "DPPO Água - Não chega encanada ao domicílio",
    "V00236_P": "DPPO Banheiro - Apenas banheiro de uso comum a mais de um domicílio",
    "V00237_P": "DPPO Banheiro - Apenas sanitário ou buraco para dejeções",
    "V00312_P": "DPPO Esgoto - Fossa rudimentar ou buraco",
    "V00313_P": "DPPO Esgoto - Vala",
    "V00314_P": "DPPO Esgoto - Rio, lago, córrego ou mar",
    "V00315_P": "DPPO Esgoto - Outra forma",
    "V00316_P": "DPPO Esgoto - Inexistente (sem banheiro nem sanitário)",
    "V00399_P": "DPPO Lixo - Queimado na propriedade",
    "V00400_P": "DPPO Lixo - Enterrado na propriedade",
    "V00401_P": "DPPO Lixo - Jogado em terreno baldio, encosta ou área pública",
    "V00402_P": "DPPO Lixo - Outro destino",
    "V00464_P": "DPPO Água - Sem ligação à rede geral de distribuição",
}

# 7. FORMATTED TABLE DISPLAY
print("\n=== PERFIL DOS GRUPOS (variáveis ordenadas pela disparidade entre os grupos) ===")

# Apply Portuguese descriptions and horizontal gradient
final_table_en = (
    df_centroids_real.rename(index=legend_dict_pt)
    .round(4)
    .style.background_gradient(cmap="Blues", axis=1, subset=mean_columns)
)

# Render formatted table
display(final_table_en)

# Reset Pandas row display configuration to default
pd.reset_option("display.max_rows")

### Perfil dos grupos em porcentagem

Médias de cada grupo nas taxas originais (% de domicílios) e lista de municípios de cada grupo.

In [ ]:
# Perfil dos grupos nas taxas originais (% de domicílios) e municípios de cada grupo
clustered_vars = [c for c in df_final.columns if c.endswith("_P")]
rates_pct = df_rates[["CD_MUN", "NM_MUN"] + clustered_vars].copy()
rates_pct["CD_MUN"] = rates_pct["CD_MUN"].astype(int)
rates_pct = rates_pct.merge(df_final[["CD_MUN", "cluster"]], on="CD_MUN", how="inner")

profile_pct = rates_pct.groupby("cluster")[clustered_vars].mean().T.round(3)
profile_pct.columns = [f"Grupo {c} - média (%)" for c in profile_pct.columns]
profile_pct["Diferença entre grupos"] = profile_pct.std(axis=1).round(3)
profile_pct.insert(0, "Descrição", profile_pct.index.map(legend_dict_pt))
profile_pct = profile_pct.sort_values("Diferença entre grupos", ascending=False)

display(profile_pct)
profile_pct.to_csv("cluster_profiles_percent.csv", encoding="utf-8-sig")

for c, group in rates_pct.groupby("cluster"):
    print(f"Grupo {c} ({len(group)} municípios): {', '.join(sorted(group['NM_MUN']))}")

### Mapa dos grupos

Cria a malha municipal unida aos grupos (usada em todos os mapas seguintes) e o mapa com os municípios de transição (silhueta negativa) hachurados.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib-scalebar"], check=False)

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch
from matplotlib_scalebar.scalebar import ScaleBar

# 1. Malha municipal do IBGE unida aos grupos (cria sp_mesh e gdf_map, usados nas células seguintes)
sp_mesh = gpd.read_file(
    "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/"
    "malhas_municipais/municipio_2022/UFs/SP/SP_Municipios_2022.zip"
)
sp_mesh["CD_MUN"] = sp_mesh["CD_MUN"].astype(int)
df_final["CD_MUN"] = df_final["CD_MUN"].astype(int)
gdf_map = sp_mesh.merge(df_final, on="CD_MUN", how="inner")

# 2. Cores, nomes dos grupos e elementos cartográficos (usados em todos os mapas)
NOMES = {0: "Grupo 1 – Carência de redes", 1: "Grupo 2 – Urbano consolidado"}
CORES = {0: "#66C2A5", 1: "#FC8D62"}
CRS_METROS = 31983  # SIRGAS 2000 / UTM 23S

def elementos_cartograficos(ax):
    """Seta do norte e barra de escala."""
    ax.add_artist(ScaleBar(1, units="m", location="lower left", box_alpha=0.8))
    ax.annotate("N", xy=(0.95, 0.95), xytext=(0.95, 0.85), xycoords="axes fraction",
                ha="center", va="center", fontsize=13, fontweight="bold",
                arrowprops=dict(facecolor="black", width=4, headwidth=12))
    ax.set_axis_off()

mun = gdf_map.to_crs(epsg=CRS_METROS)

# 3. Mapa dos grupos (municípios de transição hachurados e nomes de todos os municípios)
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Patch

m = mun.merge(sil_mun[["CD_MUN", "silhouette"]], on="CD_MUN", how="left")

fig, ax = plt.subplots(figsize=(12, 9))

# 1. Cores dos grupos
for c, sub in m.groupby("cluster"):
    sub.plot(ax=ax, color=CORES[c], edgecolor="white", linewidth=0.6)

# 2. Hachura nos municípios de transição (silhueta negativa)
transicao = m[m["silhouette"] < 0]
transicao.plot(ax=ax, facecolor="none", edgecolor="#333333", hatch="///", linewidth=0.8)

# 3. Contorno externo da RMSP
m.dissolve().boundary.plot(ax=ax, color="#333333", linewidth=0.8)

# 4. Nomes de todos os municípios
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "adjustText"], check=False)
from adjustText import adjust_text

def ponto_rotulo(geom):
    """Centro do município; se cair fora (formato irregular), usa um ponto interno."""
    c = geom.centroid
    return c if geom.contains(c) else geom.representative_point()

LIMITE_AREA_KM2 = 80          # municípios menores que isso podem ter o nome deslocado
m["area_km2"] = m.geometry.area / 1e6
fixos, moveis = [], []
for _, r in m.iterrows():
    p = ponto_rotulo(r.geometry)
    t = ax.text(p.x, p.y, r["NM_MUN"], ha="center", va="center", fontsize=7,
                fontweight="bold" if r["cluster"] == 0 else "normal", color="#1A1A1A",
                path_effects=[pe.withStroke(linewidth=2, foreground="white")])
    (moveis if r["area_km2"] < LIMITE_AREA_KM2 else fixos).append(t)

# só os nomes dos municípios pequenos são afastados; os demais ficam no centro do município
adjust_text(moveis, ax=ax, arrowprops=dict(arrowstyle="-", color="#555555", lw=0.5))

# 5. Legenda
ax.legend(handles=[Patch(facecolor=CORES[0], label=NOMES[0]),
                   Patch(facecolor=CORES[1], label=NOMES[1]),
                   Patch(facecolor="none", edgecolor="#333333", hatch="///",
                         label="Município de transição (silhueta negativa)")],
          loc="lower right", frameon=True, fontsize=10, title="Tipologias", title_fontsize=10)

elementos_cartograficos(ax)
fig.tight_layout()
plt.savefig("mapa_tipologias_RMSP.png", dpi=300, bbox_inches="tight")
plt.show()

### Densidade demográfica por grupo

In [ ]:
RAW = "https://raw.githubusercontent.com/julianevesa/rmsp-habitacao-censo2022/main/1_DATA/raw/"
basico = pd.read_csv(RAW + "ACM_basico_SP.csv", encoding="latin1", sep=";")
basico = basico[["CD_MUN", "v0001", "AREA_KM2"]].copy()   # v0001 = total de pessoas
basico["CD_MUN"] = basico["CD_MUN"].astype(int)
for col in ["v0001", "AREA_KM2"]:
    basico[col] = pd.to_numeric(basico[col].astype(str).str.replace(",", "."), errors="coerce")
basico["densidade_hab_km2"] = basico["v0001"] / basico["AREA_KM2"]

dens = basico.merge(df_final[["CD_MUN", "cluster"]], on="CD_MUN", how="inner")
resumo_dens = dens.groupby("cluster").agg(
    Municipios=("CD_MUN", "count"),
    Populacao_total=("v0001", "sum"),
    Densidade_media=("densidade_hab_km2", "mean"),
    Densidade_mediana=("densidade_hab_km2", "median"),
    Densidade_min=("densidade_hab_km2", "min"),
    Densidade_max=("densidade_hab_km2", "max"),
).round(1)
display(resumo_dens)
resumo_dens.to_csv("densidade_por_grupo.csv", encoding="utf-8-sig")

## 13. Variáveis suplementares de baixa ocorrência

Variáveis que não entraram no agrupamento por baixa ocorrência, descritas por grupo.

In [ ]:
# Supplementary low-occurrence variables: distribution by cluster (not used in the clustering)
sup_rare = df_rates[["CD_MUN", "NM_MUN"] + supplementary_vars].copy()
sup_rare["CD_MUN"] = sup_rare["CD_MUN"].astype(int)
sup_rare = sup_rare.merge(df_final[["CD_MUN", "cluster"]], on="CD_MUN", how="inner")

by_cluster = sup_rare.groupby("cluster")[supplementary_vars]
summary_rare = pd.concat(
    {"Municipalities with occurrence": by_cluster.agg(lambda x: (x > 0).sum()).T,
     "Mean (%)": by_cluster.mean().T.round(4)},
    axis=1,
)
display(summary_rare)
summary_rare.to_csv("supplementary_variables_by_cluster.csv", encoding="utf-8")

for v in supplementary_vars:
    rows = sup_rare[sup_rare[v] > 0]
    places = ", ".join(f"{n} (C{c})" for n, c in zip(rows["NM_MUN"], rows["cluster"]))
    print(f"{v}: {len(rows)} municipalities -> {places}")

### Tabela por município

Para cada condição: municípios com ocorrência, grupo, número e porcentagem de domicílios. A segunda célula gera uma versão compacta, com uma linha por condição.

In [ ]:
rows = []
for var in supplementary_vars:
    raw = var.replace("_P", "")
    occ = df_rates[df_rates[var] > 0][["CD_MUN", "NM_MUN", var]].copy()
    occ["CD_MUN"] = occ["CD_MUN"].astype(int)
    occ = occ.merge(df[["CD_MUN", raw]].astype({"CD_MUN": int}), on="CD_MUN")
    occ = occ.merge(df_final[["CD_MUN", "cluster"]], on="CD_MUN")
    for _, r in occ.iterrows():
        rows.append({"Condição": legend_dict_pt.get(var, var).split(" - ")[-1], "Município": r["NM_MUN"],
                     "Grupo": r["cluster"] + 1, "Domicílios": int(r[raw]), "% dos domicílios": round(r[var], 3)})
tabela_sup = pd.DataFrame(rows).sort_values(["Condição", "Grupo", "Domicílios"], ascending=[True, True, False])
display(tabela_sup)
tabela_sup.to_csv("tabela_variaveis_suplementares.csv", index=False, encoding="utf-8-sig")


In [ ]:
tabela_sup["Município (domicílios)"] = tabela_sup["Município"] + " (" + tabela_sup["Domicílios"].astype(str) + ")"
compacta = tabela_sup.pivot_table(index="Condição", columns="Grupo", values="Município (domicílios)",
                                  aggfunc=", ".join, fill_value="–")
compacta.columns = [f"Grupo {c}" for c in compacta.columns]
display(compacta)
compacta.to_csv("tabela_suplementares_compacta.csv", encoding="utf-8-sig")

### Mapas por condição

Um mapa para cada condição, com o número de domicílios em cada município e o contorno dos dois grupos.

In [ ]:
contorno = mun.dissolve(by="cluster").reset_index()   # contorno dos dois grupos

import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

fig, axes = plt.subplots(3, 2, figsize=(14, 17))
for ax, var in zip(axes.flat, supplementary_vars):
    raw = var.replace("_P", "")
    contagem = df[["CD_MUN", raw]].astype({"CD_MUN": int})
    mm = mun.merge(contagem, on="CD_MUN", how="left")
    mm["ocorre"] = mm[raw] > 0
    mm.plot(ax=ax, color=mm["ocorre"].map({True: "#54278F", False: "#EEEEEE"}),
            edgecolor="white", linewidth=0.3)

    # contorno dos dois grupos
    contorno[contorno["cluster"] == 0].boundary.plot(ax=ax, color="#1B7837", linewidth=1.8)
    contorno[contorno["cluster"] == 1].boundary.plot(ax=ax, color="#7F2704", linewidth=1.3, linestyle="--")

    # número de domicílios dentro de cada município com ocorrência
    for _, r in mm[mm["ocorre"]].iterrows():
        p = r.geometry.representative_point()
        ax.text(p.x, p.y, f"{int(r[raw])}", ha="center", va="center", fontsize=10,
                color="white", fontweight="bold",
                path_effects=[pe.withStroke(linewidth=2, foreground="#2D1450")])

    titulo = legend_dict_pt.get(var, var).split(" - ")[-1]
    ax.set_title(f"{titulo}\n({int(mm['ocorre'].sum())} municípios)", fontsize=13, fontweight="bold")
    ax.set_axis_off()

fig.legend(handles=[Line2D([0], [0], marker="s", color="w", markerfacecolor="#54278F", markersize=16, label="Com ocorrência (nº de domicílios)"),
                    Line2D([0], [0], marker="s", color="w", markerfacecolor="#EEEEEE", markersize=16, label="Sem ocorrência"),
                    Line2D([0], [0], color="#1B7837", linewidth=2.5, label=NOMES[0]),
                    Line2D([0], [0], color="#7F2704", linewidth=2, linestyle="--", label=NOMES[1])],
           loc="lower center", ncol=2, frameon=False, fontsize=14)
fig.subplots_adjust(left=0.01, right=0.99, top=0.95, bottom=0.07, wspace=0.03, hspace=0.12)
plt.savefig("mapas_variaveis_suplementares.png", dpi=300, bbox_inches="tight")
plt.show()

## 14. População em favelas e comunidades urbanas

Variável suplementar, que não entra no agrupamento.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# 1. Load the supplementary file generated by the database notebook (BD_)
df_sup = pd.read_csv(path + "favelas_suplementar_SP.csv", sep=";")
df_sup["CD_MUN"] = df_sup["CD_MUN"].astype(int)

# 2. Proportion of residents in favelas (over residents in permanent dwellings, V00005)
base_pop = df[["CD_MUN", "V00005"]].copy()
base_pop["CD_MUN"] = base_pop["CD_MUN"].astype(int)
df_sup = df_sup.merge(base_pop, on="CD_MUN", how="left")
df_sup["FAVELA_P"] = df_sup["N_POP_DPPO"] / df_sup["V00005"] * 100

# 3. Join with the clusters (outside the K-Means) and summarize by cluster
df_sup = df_sup.merge(df_final[["CD_MUN", "cluster"]], on="CD_MUN", how="inner")
summary_sup = df_sup.groupby("cluster")["FAVELA_P"].agg(
    Municipalities="count",
    Mean="mean",
    Median="median",
    With_favelas=lambda x: (x > 0).sum(),
).round(2)
display(summary_sup)
summary_sup.to_csv("favelas_by_cluster.csv", encoding="utf-8")

### Por município e correlação com as variáveis do agrupamento

População e proporção em favelas por município e correlação de Spearman entre a proporção em favelas e as variáveis do agrupamento.

In [ ]:
# 1. Favelas por município (população e % dos moradores), por grupo
fav_tab = df_sup[["CD_MUN", "N_POP_DPPO", "FAVELA_P", "cluster"]].merge(
    df[["CD_MUN", "NM_MUN"]].astype({"CD_MUN": int}), on="CD_MUN")
fav_tab["Grupo"] = fav_tab["cluster"] + 1
fav_tab = fav_tab.sort_values(["Grupo", "FAVELA_P"], ascending=[True, False])
display(fav_tab[["NM_MUN", "Grupo", "N_POP_DPPO", "FAVELA_P"]].round(2))
print(f"Total na RMSP: {fav_tab['N_POP_DPPO'].sum():,} | São Paulo: "
      f"{fav_tab.loc[fav_tab['NM_MUN']=='São Paulo','N_POP_DPPO'].sum():,}")

# 2. Correlação (Spearman) entre a % em favelas e as variáveis do agrupamento
corr_fav = rates_pct.merge(fav_tab[["CD_MUN", "FAVELA_P"]], on="CD_MUN")[clustered_vars + ["FAVELA_P"]] \
    .corr(method="spearman")["FAVELA_P"].drop("FAVELA_P").sort_values()
corr_fav.index = corr_fav.index.map(lambda v: legend_dict_pt.get(v, v))
display(corr_fav.round(2))

### Favelas e carências domiciliares por município

In [ ]:
cols = {"V00314_P": "Esgoto em córrego (%)", "V00050_P": "Cortiço (%)",
        "V00312_P": "Fossa rudimentar (%)", "V00113_P": "Poço raso (%)"}
cruz = fav_tab[["CD_MUN", "NM_MUN", "Grupo", "N_POP_DPPO", "FAVELA_P"]].merge(
    rates_pct[["CD_MUN"] + list(cols)], on="CD_MUN").rename(columns=cols)
display(cruz.sort_values(["Grupo", "FAVELA_P"], ascending=[True, False]).round(2))
cruz.to_csv("favelas_x_privacoes.csv", index=False, encoding="utf-8-sig")

### Mapa

Proporção de moradores em favelas (cores, em classes), número de moradores (círculos) e contorno dos dois grupos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

contorno = mun.dissolve(by="cluster").reset_index()
fav = mun.merge(df_sup[["CD_MUN", "N_POP_DPPO", "FAVELA_P"]], on="CD_MUN", how="left")
fav[["N_POP_DPPO", "FAVELA_P"]] = fav[["N_POP_DPPO", "FAVELA_P"]].fillna(0)

# 1. Classes da % de moradores em favelas (0% separado das demais)
classes = [(-0.01, 0, "Sem favelas", "#F7F7F7"),
           (0, 3, "Até 3%", "#FEE5D9"),
           (3, 10, "3% a 10%", "#FCAE91"),
           (10, 20, "10% a 20%", "#FB6A4A"),
           (20, 100, "Mais de 20%", "#CB181D")]
def classe(v):
    for lo, hi, rot, cor in classes:
        if lo < v <= hi:
            return cor
    return "#F7F7F7"
fav["cor"] = fav["FAVELA_P"].apply(classe)

fig, ax = plt.subplots(figsize=(12, 9))
fav.plot(ax=ax, color=fav["cor"], edgecolor="#BBBBBB", linewidth=0.4)

# 2. Contorno dos grupos
contorno[contorno["cluster"] == 0].boundary.plot(ax=ax, color="#1B7837", linewidth=2.2)
contorno[contorno["cluster"] == 1].boundary.plot(ax=ax, color="#7F2704", linewidth=1.6, linestyle="--")

# 3. Círculos proporcionais à população em favelas (área proporcional ao valor)
ESCALA = 0.9   # aumente ou diminua para ajustar o tamanho de todos os círculos
pts = fav[fav["N_POP_DPPO"] > 0].copy()
pts["geometry"] = pts.geometry.representative_point()
ax.scatter(pts.geometry.x, pts.geometry.y, s=np.sqrt(pts["N_POP_DPPO"]) * ESCALA,
           facecolor="none", edgecolor="#08306B", linewidth=1.2, alpha=0.9, zorder=3)

# 4. Nome dos municípios citados no texto
citados = ["São Paulo", "Mauá", "Itaquaquecetuba", "Diadema", "Embu das Artes",
           "Biritiba Mirim", "Mogi das Cruzes", "Mairiporã", "São Caetano do Sul"]
for _, r in fav[fav["NM_MUN"].isin(citados)].iterrows():
    p = r.geometry.representative_point()
    ax.annotate(r["NM_MUN"], xy=(p.x, p.y), xytext=(0, -12), textcoords="offset points",
                ha="center", fontsize=8, fontweight="bold",
                path_effects=[pe.withStroke(linewidth=2.5, foreground="white")], zorder=4)

# 5. Legendas: classes (cores), círculos e grupos
leg1 = ax.legend(handles=[Patch(facecolor=c, edgecolor="#999999", label=r) for _, _, r, c in classes],
                 title="Moradores em favelas (%)", loc="upper left", fontsize=9, title_fontsize=10)
ax.add_artist(leg1)
ref = [10_000, 100_000, 1_000_000]
leg2 = ax.legend(handles=[Line2D([0], [0], marker="o", color="w", markerfacecolor="none",
                                 markeredgecolor="#08306B", markersize=np.sqrt(np.sqrt(v) * ESCALA),
                                 label=f"{v:,}".replace(",", ".")) for v in ref],
                 title="Moradores em favelas", loc="lower left", bbox_to_anchor=(0.0, 0.08),
                 fontsize=9, title_fontsize=10, labelspacing=1.6, borderpad=1.0)
ax.add_artist(leg2)
ax.legend(handles=[Line2D([0], [0], color="#1B7837", linewidth=2.2, label=NOMES[0]),
                   Line2D([0], [0], color="#7F2704", linewidth=1.6, linestyle="--", label=NOMES[1])],
          loc="lower right", frameon=True, fontsize=10)

elementos_cartograficos(ax)
fig.tight_layout()
plt.savefig("mapa_favelas_grupos_RMSP.png", dpi=300, bbox_inches="tight")
plt.show()